# Experimental Run — GEO Audit

Pipeline experimental que mide la visibilidad GEO de programamos.es.

**Prerequisito**: Haber ejecutado `00_discover_competitors.ipynb` para tener
el vectorstore congelado en `data/frozen_vectorstore/`.

**Que hace cada run**:
1. Carga el vectorstore congelado de competidores
2. Scrapea el contenido ACTUAL de programamos.es
3. Genera embeddings solo del target y los mezcla con los congelados
4. Para cada query: retrieve → RAG Judge (JSON) → Citation Extractor
5. Guarda scorecard JSON en `data/geo/experimental/`

Ver ADR-006 en `docs/DECISIONS.md`

In [ ]:
# === Bootstrap Kaggle ===
import os, sys
from datetime import datetime
import json

if os.path.exists('/kaggle'):
    # 1. Clonar / actualizar repo (rama kaggle — contiene frozen_vectorstore)
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['GOOGLE_API_KEY'] = secrets.get_secret('GOOGLE_API_KEY')
    os.environ['OPENAI_API_KEY'] = secrets.get_secret('OPENAI_API_KEY')
    os.environ['USER_AGENT'] = 'GeoAuditBot/1.0 (TFG Research)'
    token = secrets.get_secret("github_token")

    REPO_URL = f"https://{token}@github.com/angelmanuelferrer/TFG.git"
    REPO_DIR = "/kaggle/working/TFG"
    BRANCH = "kaggle"

    if os.path.exists(REPO_DIR):
        print("Repo ya existe, haciendo pull...")
        !cd {REPO_DIR} && git checkout {BRANCH} && git pull origin {BRANCH}
    else:
        print("Clonando repo...")
        !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

    # 2. Instalar dependencias
    %pip install -q \
        "langchain>=0.3,<0.4" \
        "langchain-community>=0.3,<0.4" \
        "langchain-huggingface>=0.1,<1" \
        "langchain-openai>=0.3,<0.4" \
        "langchain-google-genai>=2.1,<3" \
        "langgraph>=0.3,<0.4" \
        "openai>=1,<2" \
        "google-genai>=1,<2" \
        "faiss-cpu>=1.9,<2" \
        "tiktoken>=0.8,<1" \
        "sentence-transformers>=3,<4" \
        "beautifulsoup4>=4,<5" \
        "requests>=2,<3" \
        "python-dotenv>=1,<2"

    # 3. Configurar entorno
    sys.path.insert(0, REPO_DIR)
    os.chdir(REPO_DIR)

    print(f"Entorno Kaggle listo — {REPO_DIR}")
else:
    from dotenv import load_dotenv
    load_dotenv()
    sys.path.insert(0, os.path.abspath('..'))

from src.config import PROJECT_ROOT
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# === Cargar configuracion ===
from src.config import load_experiment_config, get_all_queries

config = load_experiment_config()
queries = get_all_queries()
target_url = config['target_url']
target_brand = config['target_brand']

RUN_ID = datetime.now().strftime('run_%Y%m%d_%H%M%S')
print(f"Run: {RUN_ID}")
print(f"Target: {target_url}")
print(f"Queries: {len(queries)}")

In [ ]:
# === Cargar vectorstore congelado de competidores ===
from src.processing.embeddings import create_embeddings
from langchain_community.vectorstores import FAISS

embeddings = create_embeddings(config)
vs_path = PROJECT_ROOT / 'data' / 'frozen_vectorstore'

if not vs_path.exists():
    raise FileNotFoundError(
        f"Vectorstore congelado no encontrado en {vs_path}/\n"
        "Ejecuta 00_discover_competitors.ipynb primero."
    )

competitor_vs = FAISS.load_local(
    str(vs_path), embeddings, allow_dangerous_deserialization=True
)
print(f"Vectorstore congelado cargado: {competitor_vs.index.ntotal} vectores")

In [ ]:
# === Crawl contenido actual de programamos.es ===
# Mismo metodo que discovery (SiteCrawler: sitemap → BFS fallback)
# para no sesgar la comparativa target vs competidores.
from src.processing.site_crawler import SiteCrawler
from src.processing.chunker import TokenAwareChunker

crawler = SiteCrawler(config)
chunker = TokenAwareChunker(config)

target_raw_docs = crawler.crawl_site(target_url, is_target=True)

if not target_raw_docs:
    raise RuntimeError(f"No se pudo crawlear {target_url}")

target_chunks = chunker.chunk_documents(target_raw_docs)
print(f"Target: {len(target_raw_docs)} pages -> {len(target_chunks)} chunks de programamos.es")

In [ ]:
# === Merge vectores: target actual + competidores congelados ===
target_vs = FAISS.from_documents(target_chunks, embeddings)
target_vs.merge_from(competitor_vs)

print(f"Vectorstore combinado: {target_vs.index.ntotal} vectores total")

In [ ]:
# === Ejecutar RAG Judge (agent) + Citation Extractor para cada query ===
from src.rag.judge import RAGJudge
from src.rag.citation_extractor import CitationExtractor

judge = RAGJudge(config)
extractor = CitationExtractor(target_url, target_brand)

results = []
for i, query in enumerate(queries, 1):
    print(f"\n[{i}/{len(queries)}] {query}")
    
    try:
        judge_output = judge.generate_answer_with_agent(query, target_vs)
        metrics = extractor.extract_metrics(judge_output)
        
        result = {
            'query': query,
            'judge_output': judge_output,
            'metrics': metrics,
        }
        results.append(result)
        
        visible = '✓' if metrics['is_visible'] else '✗'
        print(f"  {visible} Visible={metrics['is_visible']}, SoM={metrics['som']}%, Rank={metrics['first_citation_rank']}")
    except Exception as e:
        print(f"  ERROR: {e}")
        results.append({'query': query, 'error': str(e)})

In [ ]:
# === Guardar scorecard ===
run_dir = PROJECT_ROOT / 'data' / 'geo' / 'experimental' / RUN_ID
run_dir.mkdir(parents=True, exist_ok=True)

# Scorecard resumen
valid_results = [r for r in results if 'metrics' in r]
visible_count = sum(1 for r in valid_results if r['metrics']['is_visible'])
ranked_results = [r for r in valid_results if r['metrics']['first_citation_rank'] is not None]

scorecard = {
    'run_id': RUN_ID,
    'timestamp': datetime.now().isoformat(),
    'target_url': target_url,
    'target_brand': target_brand,
    'n_queries': len(queries),
    'n_successful': len(valid_results),
    'aggregate_metrics': {
        'visibility_rate': (visible_count / len(valid_results) * 100) if valid_results else 0,
        'avg_som': sum(r['metrics']['som'] for r in valid_results) / len(valid_results) if valid_results else 0,
        'avg_rank': sum(r['metrics']['first_citation_rank'] for r in ranked_results) / len(ranked_results) if ranked_results else None,
    },
    'token_usage': judge.get_token_usage_summary(),
    'config_snapshot': config,
}

# Guardar archivos
with open(run_dir / 'scorecard.json', 'w', encoding='utf-8') as f:
    json.dump(scorecard, f, ensure_ascii=False, indent=2)

with open(run_dir / 'raw_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Resultados guardados en {run_dir}/")
print(f"\n=== RESUMEN ===")
print(f"Visibilidad: {scorecard['aggregate_metrics']['visibility_rate']:.1f}%")
print(f"SoM promedio: {scorecard['aggregate_metrics']['avg_som']:.1f}%")
avg_rank = scorecard['aggregate_metrics']['avg_rank']
print(f"Rank promedio: {f'{avg_rank:.1f}' if avg_rank else 'N/A (no citado)'}")
print(f"Tokens usados: {scorecard['token_usage']}")

In [ ]:
# === Detalle por query ===
print(f"{'Query':<70} {'Vis':>4} {'SoM':>6} {'Rank':>5}")
print('-' * 90)
for r in results:
    if 'metrics' in r:
        m = r['metrics']
        vis = 'SI' if m['is_visible'] else 'NO'
        rank = str(m['first_citation_rank']) if m['first_citation_rank'] else '-'
        print(f"{r['query'][:68]:<70} {vis:>4} {m['som']:>5.1f}% {rank:>5}")
    else:
        print(f"{r['query'][:68]:<70} {'ERR':>4}")